In [434]:
from game import Game
from enum import Enum
import time
from queue import PriorityQueue
import heapq 
from collections import deque
import time
import os
import re

## Load maps

In [435]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [436]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [437]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [438]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [439]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [440]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [441]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [442]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


In [443]:
movement = ['U', 'D', 'R', 'L']

class State :
    def __init__(self):
        self.player_pose = tuple()
        self.box_positions = tuple()

    def get_box_positions(self):
        return self.box_positions
    def get_player_position(self):
        return self.player_pose
    def set_box_positions (self, box_poses):
        self.box_positions = box_poses
    def set_player_position (self, player_pos):
        self.player_pose = player_pos
    def __hash__(self):
        return hash((self.player_pose, self.box_positions))

    def __eq__(self, other): 
        if not isinstance(other, State):  
            return False  
        return self.player_pose == other.player_pose and self.box_positions == other.box_positions  
    def get_child_nodes(self, game, path):
        childs = set()
        game.set_box_positions(self.box_positions)
        game.set_player_position(self.player_pose)

        for direction in movement:
            if game.apply_move(direction):
                node = State()
                node.set_player_position(tuple(game.get_player_position()))
                node.set_box_positions(tuple(game.get_box_locations()))
                childs.add(((path + direction), node))

            game.set_box_positions(self.box_positions)
            game.set_player_position(self.player_pose)
        
        return childs

            
def is_game_won(boxes, goals):
    for i in range(len(boxes)):
        if boxes[i] != goals[i]:
            return False
    return True
  
def generate_initial_state(game):
    root = State()
    root.set_box_positions(tuple(game.get_box_locations()))
    root.set_player_position(tuple(game.get_player_position()))
    return (root)

def generate_visited_element(child):
    return (tuple(child.get_player_position()), tuple((child.get_box_positions())))

## Solvers

### BFS

In [444]:
def solver_bfs(game_map):
    game = Game(game_map)
    goal_locations = game.get_goal_locations()
    visited = set()
    fringe = deque()

    root = generate_initial_state(game)
    fringe.append(("",root))
    visited.add(root)
    
    while fringe:
        (path , state) = fringe.popleft()

        if (is_game_won(state.get_box_positions(), goal_locations)):
            return path, len(visited)
              
        childs = state.get_child_nodes(game,path)

        for child_path, child_state in childs: 
            
            if child_state not in visited:
                fringe.append((child_path, child_state))
                visited.add(child_state)

    return None, 0

### DFS

In [445]:
def solver_dfs(game_map):
    game = Game(game_map)  
    goal_locations = game.get_goal_locations()  
    visited = set()
    fringe = deque()

    root = generate_initial_state(game)
    fringe.append(("", root))

    while fringe:
        path, state = fringe.pop()

        if state in visited:
            continue

        if is_game_won(state.get_box_positions(), goal_locations):
            return path, len(visited)
        
        childs = state.get_child_nodes(game, path)
        visited.add(state)

        for child in childs:
            fringe.append(child)

    return None, 0

### IDS

In [446]:
def solver_ids(game_map):
    game = Game(game_map)
    goal_locations = game.get_goal_locations()
    max_depth = 0
    search_go = False

    def dfs(root):
        nonlocal search_go
        visited = set()
        level = dict()
        stack = deque()

        stack.append(("",root))
        visited.add(root)
        level[root] = 0

        search_go = True
        while stack:
            (path , state) = stack.pop()
            visited_level = level[state]

            if visited_level < len(path):
                continue

            if len(path) < max_depth:
                childs = state.get_child_nodes(game,path)

                for child_path, child_state in childs:
                    if child_state not in visited:
                        if is_game_won(child_state.get_box_positions(), goal_locations):
                            return child_path, len(visited), True
                        stack.append((child_path, child_state))
                        visited.add(child_state)
                        level[child_state] = len(child_path)
                    else:
                        visited_level = level[child_state]
                        if visited_level > len(child_path):
                            stack.append((child_path, child_state))
                            level[child_state] = len(child_path)
            else:
                search_go = False


        return None, 0, False
    
    root = generate_initial_state(game)
    while not search_go:
        path, num_visited, game_won = dfs(root)
        if game_won:
            return path, num_visited
        max_depth += 1

    return None, 0

### A*

In [447]:
def manhattan_dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def calculate_best_portal_path(start, end, portals):
    best_distance = manhattan_dist(start, end)

    for portal in portals:
        best_distance = min(best_distance,
            manhattan_dist(start, portal[0]) + manhattan_dist(portal[1], end),
            manhattan_dist(start, portal[1]) + manhattan_dist(portal[0], end)
        )

    if len(portals) > 1:
        portal_combinations = [
            (0, 0, 1, 0), (0, 0, 1, 1),
            (0, 1, 1, 0), (0, 1, 1, 1)
        ]
        for a, b, c, d in portal_combinations:
            best_distance = min(best_distance,
                manhattan_dist(start, portals[a][b]) +
                manhattan_dist(portals[a][1-b], portals[c][d]) +
                manhattan_dist(portals[c][1-d], end)
            )

    return best_distance

def heuristic(game: Game, naive=False):
    player_loc = game.get_player_position()
    boxes_loc = game.get_box_locations()
    portals_loc = game.get_portal_locations()
    goals_loc = game.get_goal_locations()

    if naive:
        return sum(manhattan_dist(box, goal) for box, goal in zip(boxes_loc, goals_loc))

    box_to_goal_distances = [
        calculate_best_portal_path(box, goal, portals_loc)
        for box, goal in zip(boxes_loc, goals_loc)
    ]
    player_to_box_distances = [
        calculate_best_portal_path(player_loc, box, portals_loc)
        for box in boxes_loc
    ]

    return sum(box_to_goal_distances) + min(player_to_box_distances)

def heuristic_old(game:Game):
    player_loc = game.get_player_position()
    boxes_loc = game.get_box_locations()
    goals_loc = game.get_goal_locations()
    
    heuristic_val = sum([manhattan_dist(boxes_loc[i], goals_loc[i]) for i in range(len(boxes_loc))])
    for box in boxes_loc:
        player_to_box = manhattan_dist(player_loc, box)
        heuristic_val += player_to_box
    
    return heuristic_val



def solver_astar(game_map, heuristic_func=heuristic_old, weight=20):
    game = Game(game_map)
    pq = []
    explored = set()
    g_n_dict = dict()

    root = generate_initial_state(game)
    heapq.heappush(pq, ((weight * heuristic_func(game)), ("", root)))
    g_n_dict[root] = 0

    while pq:
        F_n, (path, state) = heapq.heappop(pq)

        game.set_box_positions(state.get_box_positions())
        game.set_player_position(state.get_player_position())

        if game.is_game_won():
            return path, len(explored)

        g_n = g_n_dict[state] + 1

        for direction in movement:
            if game.apply_move(direction):
                node = State()
                node.set_player_position(tuple(game.get_player_position()))
                node.set_box_positions(tuple(game.get_box_locations()))

                if node in g_n_dict and g_n >= g_n_dict[node]:
                    game.set_box_positions(state.get_box_positions())
                    game.set_player_position(state.get_player_position())
                    continue

                g_n_dict[node] = g_n
                heapq.heappush(pq, ((g_n + (weight * heuristic_func(game))), ((path + direction), node)))
                explored.add(node)

            game.set_box_positions(state.get_box_positions())
            game.set_player_position(state.get_player_position())

    return None, len(explored)


## Solve

In [448]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar
}

In [449]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [450]:
# for i in range(1, 11):
#     if i != 9:
#         solve(i, "A*")
#     else:
#         print(f"for map {i},we have big time to find solution........")
solve(9, "A*")

KeyboardInterrupt: 

In [ ]:
def solve_all():
    for map in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map, method)
            

In [ ]:
# solve_all() # Solve all maps using all methods